# Customer Churn Prediction Using Logistic Regression

This notebook builds an end-to-end machine learning pipeline to predict customer churn probability using the supplied customer churn dataset.

**Pipeline:** Data inspection → cleaning → One-Hot Encoding → MinMax Scaling → 80/20 split → Logistic Regression → evaluation → churn risk scores.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay
)

## 2. Load the Dataset

The notebook uses the project dataset stored in `data/customer_churn_sample.csv`.

In [ ]:
df = pd.read_csv('../data/customer_churn_sample.csv')

print('Dataset shape:', df.shape)
display(df.head())

## 3. Data Inspection and Cleaning

In [ ]:
print('Missing values:')
display(df.isnull().sum())

print('Duplicate Customer IDs:', df['CustomerID'].duplicated().sum())

df = df.drop_duplicates(subset=['CustomerID']).copy()
            df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print('Cleaned shape:', df.shape)

## 4. Separate Features and Target

In [ ]:
customer_ids = df['CustomerID'].copy()
X = df.drop(columns=['CustomerID', 'Churn'])
y = df['Churn']

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

## 5. Build Preprocessing Pipeline

- Numerical features: median imputation + MinMax scaling
- Categorical features: most-frequent imputation + One-Hot Encoding

In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])

## 6. Split Data 80/20

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))

## 7. Train Logistic Regression Model

In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print('Model training completed.')

## 8. Model Evaluation

In [ ]:
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)

print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-Score:  {f1:.4f}')
print(f'ROC-AUC:   {roc_auc:.4f}')

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn'], zero_division=0))

## 9. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(cm, display_labels=['No Churn', 'Churn'])
disp.plot()
plt.title('Confusion Matrix — Customer Churn')
plt.show()

## 10. ROC-AUC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-AUC Curve — Customer Churn')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 11. Generate Customer Churn Risk Scores

In [ ]:
results = pd.DataFrame({
    'CustomerID': customer_ids.loc[X_test.index].values,
    'Actual_Churn': y_test.values,
    'Predicted_Churn': y_pred,
    'Churn_Probability': np.round(y_prob, 4)
})

results['Churn_Risk'] = pd.cut(
    results['Churn_Probability'],
    bins=[-0.001, 0.30, 0.60, 1.00],
    labels=['Low', 'Medium', 'High']
)

results = results.sort_values('Churn_Probability', ascending=False)
display(results)

## 12. Export Predictions

The exported file can be used as a retention-risk output for further analysis or dashboards.

In [ ]:
results.to_csv('../outputs/churn_predictions.csv', index=False)

print('Saved: outputs/churn_predictions.csv')

## 13. Business Interpretation

The model produces a probability between 0 and 1 for each test customer. The probability is converted into three illustrative risk bands:

- **Low:** below 30%
- **Medium:** 30% to below 60%
- **High:** 60% or above

These thresholds are project-defined and should be tuned using business costs and validation data in a production system.

### Dataset limitation

This supplied dataset contains only 15 records. Therefore, the evaluation results can change substantially with a different split and should not be treated as evidence of production-level model performance. A larger dataset and cross-validation would be required for reliable deployment.